# Fine-tune TrOCR-Large on full IAM-line (Kaggle)

Same recipe as the local `finetune_trocr.py` / Colab notebook attempts, adapted for Kaggle:
- Before running: **Settings (right sidebar) -> Accelerator -> GPU T4 x2** (or P100), and **Internet -> On** (needed for pip installs + downloading IAM data). Both are off by default on Kaggle.
- Uses `microsoft/trocr-large-handwritten`, `lr=2e-5` (fixes the CER regression seen at the script's `5e-5` default), freeze_encoder=6/freeze_decoder=12 (scaled for large's 24/24-layer stacks).
- Pushes straight to the Hugging Face Hub at the end -- no Drive/download round-trip this time (that failed twice on Colab with a ~2.2GB checkpoint).

In [ ]:
!pip install -q -U transformers jiwer sentencepiece huggingface_hub
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Download the full IAM-line dataset (train + validation)

In [ ]:
import csv, io, os, urllib.request
import pandas as pd
from PIL import Image

DATA_DIR = '/kaggle/working/data'
LINES_DIR = os.path.join(DATA_DIR, 'lines')
os.makedirs(LINES_DIR, exist_ok=True)

PARQUET_URLS = {
    'train': 'https://huggingface.co/datasets/Teklia/IAM-line/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet',
    'val': 'https://huggingface.co/datasets/Teklia/IAM-line/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet',
}

def write_split(df, split_name, csv_path):
    rows_written = 0
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['image_path', 'text'])
        for i, row in df.iterrows():
            text = row['text']
            if not isinstance(text, str) or not text.strip():
                continue
            img_data = row['image']
            img_bytes = img_data.get('bytes', b'') if isinstance(img_data, dict) else img_data
            image = Image.open(io.BytesIO(img_bytes)).convert('RGB')
            filename = f'{split_name}_{i:05d}.png'
            image.save(os.path.join(LINES_DIR, filename))
            writer.writerow([os.path.join('lines', filename), text])
            rows_written += 1
    print(f'{split_name}: {rows_written} lines -> {csv_path}')

for split_name, url in PARQUET_URLS.items():
    parquet_path = os.path.join(DATA_DIR, f'_temp_{split_name}.parquet')
    print(f'Downloading {split_name} parquet...')
    urllib.request.urlretrieve(url, parquet_path)
    df = pd.read_parquet(parquet_path)
    csv_name = 'train.csv' if split_name == 'train' else 'val.csv'
    write_split(df, split_name, os.path.join(DATA_DIR, csv_name))
    os.remove(parquet_path)

print('Data ready.')

## 2. Fine-tuning (TrOCR-Large, tuned LR + freeze ratio)

In [ ]:
import random, cv2
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    TrOCRProcessor, VisionEncoderDecoderModel, RobertaTokenizer, ViTImageProcessor,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
)
import jiwer

BASE_MODEL = 'microsoft/trocr-large-handwritten'

def elastic_distort(image_np, alpha=34, sigma=4, seed=None):
    rng = np.random.RandomState(seed)
    shape = image_np.shape[:2]
    dx = cv2.GaussianBlur((rng.rand(*shape) * 2 - 1), (0, 0), sigma) * alpha
    dy = cv2.GaussianBlur((rng.rand(*shape) * 2 - 1), (0, 0), sigma) * alpha
    x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    map_x = (x + dx).astype(np.float32)
    map_y = (y + dy).astype(np.float32)
    return cv2.remap(image_np, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderValue=(255, 255, 255))

def random_rotate(image_np, max_angle=3.0):
    angle = random.uniform(-max_angle, max_angle)
    h, w = image_np.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image_np, matrix, (w, h), borderValue=(255, 255, 255))

def augment(pil_image):
    arr = np.array(pil_image.convert('RGB'))
    arr = random_rotate(arr)
    if random.random() < 0.5:
        arr = elastic_distort(arr)
    return Image.fromarray(arr)

class LineOCRDataset(Dataset):
    def __init__(self, csv_path, processor, image_root=None, max_target_length=128, train=True):
        self.processor, self.max_target_length, self.train = processor, max_target_length, train
        self.image_root = image_root or os.path.dirname(os.path.abspath(csv_path))
        with open(csv_path, newline='', encoding='utf-8') as f:
            self.rows = list(csv.DictReader(f))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        image_path = row['image_path']
        if not os.path.isabs(image_path):
            image_path = os.path.join(self.image_root, image_path)
        image = Image.open(image_path).convert('RGB')
        if self.train:
            image = augment(image)
        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(row['text'], padding='max_length', max_length=self.max_target_length, truncation=True).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

def freeze_layers(model, freeze_encoder_layers, freeze_decoder_layers):
    if freeze_encoder_layers > 0:
        for param in model.encoder.embeddings.parameters():
            param.requires_grad = False
        for layer in model.encoder.layers[:freeze_encoder_layers]:
            for param in layer.parameters():
                param.requires_grad = False
    if freeze_decoder_layers > 0:
        decoder_layers = model.decoder.model.decoder.layers
        for layer in decoder_layers[:freeze_decoder_layers]:
            for param in layer.parameters():
                param.requires_grad = False

def build_compute_metrics(processor):
    def compute_metrics(eval_pred):
        pred_ids, label_ids = eval_pred.predictions, eval_pred.label_ids
        label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
        pred_ids = np.where(pred_ids >= 0, pred_ids, processor.tokenizer.pad_token_id)
        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
        pred_str = [p if p.strip() else ' ' for p in pred_str]
        label_str = [l if l.strip() else ' ' for l in label_str]
        return {'cer': jiwer.cer(label_str, pred_str), 'wer': jiwer.wer(label_str, pred_str)}
    return compute_metrics

print(f'Loading {BASE_MODEL} ...')
image_processor = ViTImageProcessor.from_pretrained(BASE_MODEL)
tokenizer = RobertaTokenizer.from_pretrained(BASE_MODEL)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

freeze_layers(model, freeze_encoder_layers=6, freeze_decoder_layers=12)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

train_dataset = LineOCRDataset(f'{DATA_DIR}/train.csv', processor, train=True)
eval_dataset = LineOCRDataset(f'{DATA_DIR}/val.csv', processor, train=False)
print(f'Train lines: {len(train_dataset)} | Eval lines: {len(eval_dataset)}')

OUTPUT_DIR = '/kaggle/working/out/trocr-large-finetuned-v1'
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=10,
    report_to=[],
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    compute_metrics=build_compute_metrics(processor),
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
metrics = trainer.evaluate()
print(f"Final eval CER: {metrics.get('eval_cer'):.4f}  WER: {metrics.get('eval_wer'):.4f}")

## 3. Push straight to the Hugging Face Hub
Best done via a Kaggle **Secret** rather than pasting the token in plaintext: Add-ons -> Secrets -> add `HF_TOKEN` -> attach it to this notebook. Then it's read from the environment below.

Get a **write** token first: https://huggingface.co/settings/tokens -> New token -> role "Write".

In [ ]:
from huggingface_hub import login
import os

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = 'PASTE_YOUR_HF_WRITE_TOKEN_HERE'  # fallback if you didn't set up a Kaggle Secret

login(token=hf_token)

HF_REPO_ID = 'your-hf-username/trocr-large-iam-finetuned'  # <-- edit this

model.push_to_hub(HF_REPO_ID)
processor.push_to_hub(HF_REPO_ID)
print(f'Pushed to https://huggingface.co/{HF_REPO_ID}')
print('Evaluate locally with: python evaluate_model.py --model ' + HF_REPO_ID + ' --csv data/val.csv --image-root data')